# 🛒 Superstore Sales Analysis — SQL Business Intelligence
### SQL | SQLite | Python | RFM Customer Segmentation

---

## 📌 Business Problem

A U.S. retail superstore operating across four regions (West, East, Central, South) wants to better understand four years of sales data (2015–2018). Leadership needs answers to critical questions:

- Which regions and segments are driving revenue growth?
- Which products and sub-categories are top performers?
- Are we retaining customers or losing them after the first purchase?
- How fast are we shipping, and does it vary by region?

This analysis goes beyond standard descriptive queries by implementing a **full RFM (Recency, Frequency, Monetary) scoring model entirely in SQL** — a technique used by real retail analytics teams to segment customers and prioritize marketing spend.

---

## 🗄️ Database Schema

```
orders (9,800 line items)       customers (793)        products (1,861)
─────────────────────────       ───────────────        ────────────────
row_id                          customer_id (PK)       product_id (PK)
order_id                        customer_name          product_name
order_date                      segment                category
ship_date                       city                   sub_category
ship_mode                       state
customer_id (FK)                region
product_id  (FK)
sales
days_to_ship
```

---

## 🔧 SQL Concepts Used
- `SELECT`, `WHERE`, `GROUP BY`, `ORDER BY`, `HAVING`
- `INNER JOIN`, `LEFT JOIN`
- Subqueries & CTEs (`WITH` clause — multi-step)
- Window Functions: `RANK()`, `DENSE_RANK()`, `LAG()`, `SUM() OVER()`, `NTILE()`
- `CASE WHEN` statements
- Date functions: `strftime()`, `julianday()`
- **RFM scoring model built entirely in SQL**

---

## 0. Database Setup
*This cell builds the SQLite database from the raw CSV file. Run it once — it creates `superstore.db` in your working directory. After that, all SQL queries in this notebook run directly against that database.*

In [ ]:
import sqlite3
import pandas as pd

# Load the raw CSV
df = pd.read_csv('train.csv', encoding='latin-1')
print(f'Raw data loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')

# Parse date columns
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  dayfirst=True)

# Engineer shipping days feature
df['Days to Ship'] = (df['Ship Date'] - df['Order Date']).dt.days

# Format dates as strings for SQLite compatibility
df['Order Date'] = df['Order Date'].dt.strftime('%Y-%m-%d')
df['Ship Date']  = df['Ship Date'].dt.strftime('%Y-%m-%d')

# Standardize column names: lowercase, underscores
df.columns = [
    c.strip().lower().replace(' ', '_').replace('-', '_')
    for c in df.columns
]

# Connect to SQLite (creates the file if it doesn't exist)
conn = sqlite3.connect('superstore.db')

# --- orders table: one row per line item ---
df.to_sql('orders', conn, if_exists='replace', index=False)

# --- customers dimension table: one row per unique customer ---
customers = (
    df[['customer_id','customer_name','segment','city','state','region']]
    .drop_duplicates(subset='customer_id')
    .reset_index(drop=True)
)
customers.to_sql('customers', conn, if_exists='replace', index=False)

# --- products dimension table: one row per unique product ---
products = (
    df[['product_id','product_name','category','sub_category']]
    .drop_duplicates(subset='product_id')
    .reset_index(drop=True)
)
products.to_sql('products', conn, if_exists='replace', index=False)

conn.close()

print('\nDatabase built successfully: superstore.db')
print(f'  orders table:    {len(df):,} rows')
print(f'  customers table: {len(customers):,} unique customers')
print(f'  products table:  {len(products):,} unique products')
print('\nSchema:')
print('  orders    -> order_id, customer_id, product_id, order_date, ship_date, ship_mode, sales, days_to_ship')
print('  customers -> customer_id, customer_name, segment, city, state, region')
print('  products  -> product_id, product_name, category, sub_category')

## 0. Setup — Connect to Database

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

conn = sqlite3.connect('superstore.db')

def run_query(sql):
    """Execute a SQL query and return results as a DataFrame."""
    return pd.read_sql_query(sql, conn)

print('Connected to superstore.db')
for t in ['orders', 'customers', 'products']:
    n = run_query(f'SELECT COUNT(*) AS n FROM {t}')['n'][0]
    print(f'  {t}: {n:,} rows')

---
## 📦 Section 1 — Sales Performance
> *Understanding which regions, segments, and time periods generate the most revenue is the foundation of retail business analysis.*
---
### Query 1.1 — Overall Business Summary

In [ ]:
q1_1 = run_query("""
    SELECT
        COUNT(DISTINCT order_id)                        AS total_orders,
        COUNT(DISTINCT customer_id)                     AS unique_customers,
        COUNT(*)                                        AS total_line_items,
        ROUND(SUM(sales), 2)                            AS total_revenue,
        ROUND(AVG(sales), 2)                            AS avg_line_item_value,
        ROUND(SUM(sales) / COUNT(DISTINCT order_id), 2) AS avg_order_value
    FROM orders
""")

print('=== Overall Business Summary (2015-2018) ===')
for col in q1_1.columns:
    val = q1_1[col][0]
    if any(k in col for k in ['revenue','value']):
        print(f'  {col:<28} ${val:>12,.2f}')
    else:
        print(f'  {col:<28} {val:>12,.0f}')

### Query 1.2 — Revenue by Region

In [ ]:
q1_2 = run_query("""
    SELECT
        c.region,
        COUNT(DISTINCT o.order_id)                       AS total_orders,
        COUNT(DISTINCT o.customer_id)                    AS unique_customers,
        ROUND(SUM(o.sales), 2)                           AS total_revenue,
        ROUND(SUM(o.sales) / COUNT(DISTINCT o.order_id), 2) AS avg_order_value
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.region
    ORDER BY total_revenue DESC
""")

print(q1_2.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Revenue Performance by Region', fontsize=14, fontweight='bold')
colors = sns.color_palette('Blues_d', len(q1_2))

bars = axes[0].bar(q1_2['region'], q1_2['total_revenue'], color=colors, edgecolor='white')
axes[0].set_title('Total Revenue by Region')
axes[0].set_ylabel('Revenue ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for bar, val in zip(bars, q1_2['total_revenue']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
                 f'${val:,.0f}', ha='center', fontsize=9)

bars2 = axes[1].bar(q1_2['region'], q1_2['avg_order_value'], color=colors, edgecolor='white')
axes[1].set_title('Avg Order Value by Region')
axes[1].set_ylabel('Avg Order Value ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for bar, val in zip(bars2, q1_2['avg_order_value']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'${val:,.0f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

### Query 1.3 — Monthly Revenue Trend with Month-over-Month Growth
*Uses `LAG()` window function to compare each month's revenue to the previous month*

In [ ]:
q1_3 = run_query("""
    WITH monthly AS (
        SELECT
            strftime('%Y-%m', order_date) AS year_month,
            ROUND(SUM(sales), 2)          AS monthly_revenue
        FROM orders
        GROUP BY year_month
    )
    SELECT
        year_month,
        monthly_revenue,
        LAG(monthly_revenue) OVER (ORDER BY year_month) AS prev_month_revenue,
        ROUND(
            (monthly_revenue - LAG(monthly_revenue) OVER (ORDER BY year_month))
            / LAG(monthly_revenue) OVER (ORDER BY year_month) * 100
        , 1) AS mom_growth_pct
    FROM monthly
    ORDER BY year_month
""")

print('Monthly Revenue with MoM Growth (first 12 months shown):')
print(q1_3.head(12).to_string(index=False))

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.fill_between(range(len(q1_3)), q1_3['monthly_revenue'], alpha=0.15, color='steelblue')
ax1.plot(range(len(q1_3)), q1_3['monthly_revenue'], color='steelblue', linewidth=2, marker='o', markersize=3)
ax1.set_title('Monthly Revenue Trend with MoM Growth (2015-2018)', fontsize=13, fontweight='bold')
ax1.set_ylabel('Monthly Revenue ($)', color='steelblue')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
step = max(1, len(q1_3) // 12)
ax1.set_xticks(range(0, len(q1_3), step))
ax1.set_xticklabels(q1_3['year_month'].iloc[::step], rotation=45, ha='right', fontsize=8)

ax2 = ax1.twinx()
mom = q1_3['mom_growth_pct'].fillna(0)
ax2.bar(range(len(q1_3)), mom,
        color=['#2ecc71' if v >= 0 else '#e74c3c' for v in mom],
        alpha=0.4, width=0.6)
ax2.set_ylabel('MoM Growth (%)', color='gray')
ax2.axhline(y=0, color='gray', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.show()

### Query 1.4 — Year-over-Year Revenue by Region
*Uses `CASE WHEN` to pivot years into columns for direct side-by-side comparison*

In [ ]:
q1_4 = run_query("""
    SELECT
        c.region,
        ROUND(SUM(CASE WHEN strftime('%Y', o.order_date) = '2015' THEN o.sales ELSE 0 END), 2) AS rev_2015,
        ROUND(SUM(CASE WHEN strftime('%Y', o.order_date) = '2016' THEN o.sales ELSE 0 END), 2) AS rev_2016,
        ROUND(SUM(CASE WHEN strftime('%Y', o.order_date) = '2017' THEN o.sales ELSE 0 END), 2) AS rev_2017,
        ROUND(SUM(CASE WHEN strftime('%Y', o.order_date) = '2018' THEN o.sales ELSE 0 END), 2) AS rev_2018,
        ROUND(
            (SUM(CASE WHEN strftime('%Y', o.order_date) = '2018' THEN o.sales ELSE 0 END) -
             SUM(CASE WHEN strftime('%Y', o.order_date) = '2017' THEN o.sales ELSE 0 END))
            / SUM(CASE WHEN strftime('%Y', o.order_date) = '2017' THEN o.sales ELSE 0 END) * 100
        , 1) AS yoy_growth_pct
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.region
    ORDER BY rev_2018 DESC
""")

print('Year-over-Year Revenue by Region:')
print(q1_4.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(q1_4))
width = 0.2
years_cols = ['rev_2015','rev_2016','rev_2017','rev_2018']
palette = ['#aec6cf','#6fa8d6','#3b82c4','#1a4f8a']
for i, (yr, col) in enumerate(zip(years_cols, palette)):
    ax.bar([xi + i*width for xi in x], q1_4[yr], width,
           label=yr.replace('rev_',''), color=col, edgecolor='white')
ax.set_title('Year-over-Year Revenue by Region (2015-2018)', fontsize=13, fontweight='bold')
ax.set_xticks([xi + 1.5*width for xi in x])
ax.set_xticklabels(q1_4['region'])
ax.set_ylabel('Revenue ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(title='Year')
plt.tight_layout()
plt.show()

### Query 1.5 — Revenue by Customer Segment

In [ ]:
q1_5 = run_query("""
    SELECT
        c.segment,
        COUNT(DISTINCT o.order_id)                           AS total_orders,
        COUNT(DISTINCT c.customer_id)                        AS unique_customers,
        ROUND(SUM(o.sales), 2)                               AS total_revenue,
        ROUND(SUM(o.sales) / COUNT(DISTINCT c.customer_id), 2) AS revenue_per_customer
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.segment
    ORDER BY total_revenue DESC
""")

print(q1_5.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Revenue by Customer Segment', fontsize=14, fontweight='bold')
palette = sns.color_palette('Set2', 3)

axes[0].pie(q1_5['total_revenue'], labels=q1_5['segment'],
            autopct='%1.1f%%', colors=palette, startangle=90)
axes[0].set_title('Revenue Share by Segment')

bars = axes[1].bar(q1_5['segment'], q1_5['revenue_per_customer'], color=palette, edgecolor='white')
axes[1].set_title('Revenue per Customer by Segment')
axes[1].set_ylabel('Revenue per Customer ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for bar, val in zip(bars, q1_5['revenue_per_customer']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'${val:,.0f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## 📦 Section 2 — Product Performance
> *Knowing which categories and sub-categories drive the most revenue helps merchandising and buying teams make smarter inventory decisions.*
---
### Query 2.1 — Revenue by Category & Sub-Category

In [ ]:
q2_1 = run_query("""
    SELECT
        p.category,
        p.sub_category,
        COUNT(DISTINCT o.order_id)                           AS total_orders,
        ROUND(SUM(o.sales), 2)                               AS total_revenue,
        ROUND(AVG(o.sales), 2)                               AS avg_sale_value,
        ROUND(SUM(o.sales) * 100.0
              / SUM(SUM(o.sales)) OVER (), 1)                AS pct_of_total_revenue
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    GROUP BY p.category, p.sub_category
    ORDER BY p.category, total_revenue DESC
""")

print(q2_1.to_string(index=False))

q2_1_sorted = q2_1.sort_values('total_revenue')
cat_colors = {'Furniture': '#4C72B0', 'Office Supplies': '#55A868', 'Technology': '#DD8452'}
bar_colors = [cat_colors[c] for c in q2_1_sorted['category']]

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(q2_1_sorted['sub_category'], q2_1_sorted['total_revenue'],
               color=bar_colors, edgecolor='white')
ax.set_title('Revenue by Sub-Category', fontsize=13, fontweight='bold')
ax.set_xlabel('Total Revenue ($)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for bar, val in zip(bars, q2_1_sorted['total_revenue']):
    ax.text(bar.get_width() + 1000, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}', va='center', fontsize=8)
legend_handles = [mpatches.Patch(facecolor=v, label=k) for k, v in cat_colors.items()]
ax.legend(handles=legend_handles, title='Category')
plt.tight_layout()
plt.show()

### Query 2.2 — Top 10 Best-Selling Products

In [ ]:
q2_2 = run_query("""
    SELECT
        p.product_name,
        p.category,
        p.sub_category,
        COUNT(DISTINCT o.order_id)  AS times_ordered,
        ROUND(SUM(o.sales), 2)      AS total_revenue
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    GROUP BY p.product_id, p.product_name, p.category, p.sub_category
    ORDER BY total_revenue DESC
    LIMIT 10
""")

print('Top 10 Best-Selling Products by Revenue:')
print(q2_2.to_string(index=False))

labels = [name[:45] + '...' if len(name) > 45 else name for name in q2_2['product_name']]
fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(labels[::-1], q2_2['total_revenue'][::-1],
               color=sns.color_palette('Blues_d', len(q2_2)), edgecolor='white')
ax.set_title('Top 10 Products by Revenue', fontsize=13, fontweight='bold')
ax.set_xlabel('Total Revenue ($)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

### Query 2.3 — Pareto Analysis: Which Sub-Categories Drive 80% of Revenue?
*Uses cumulative `SUM() OVER()` to identify the vital few sub-categories that account for the majority of sales*

In [ ]:
q2_3 = run_query("""
    WITH sub_rev AS (
        SELECT
            p.sub_category,
            p.category,
            ROUND(SUM(o.sales), 2) AS total_revenue
        FROM orders o
        JOIN products p ON o.product_id = p.product_id
        GROUP BY p.sub_category, p.category
    ),
    grand AS (SELECT SUM(total_revenue) AS total FROM sub_rev)
    SELECT
        s.sub_category,
        s.category,
        s.total_revenue,
        ROUND(s.total_revenue / g.total * 100, 1) AS pct_of_total,
        ROUND(
            SUM(s.total_revenue) OVER (
                ORDER BY s.total_revenue DESC
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) / g.total * 100
        , 1) AS cumulative_pct
    FROM sub_rev s, grand g
    ORDER BY s.total_revenue DESC
""")

print('Pareto Analysis — Cumulative Revenue by Sub-Category:')
print(q2_3.to_string(index=False))

fig, ax1 = plt.subplots(figsize=(13, 5))
x = range(len(q2_3))
bar_colors = [cat_colors[c] for c in q2_3['category']]
ax1.bar(x, q2_3['pct_of_total'], color=bar_colors, edgecolor='white', alpha=0.85)
ax1.set_ylabel('% of Total Revenue')
ax1.set_xticks(list(x))
ax1.set_xticklabels(q2_3['sub_category'], rotation=45, ha='right', fontsize=9)
ax1.set_title('Pareto Analysis — Cumulative Revenue Contribution by Sub-Category',
              fontsize=13, fontweight='bold')

ax2 = ax1.twinx()
ax2.plot(list(x), q2_3['cumulative_pct'], color='red', linewidth=2, marker='o', markersize=4)
ax2.axhline(y=80, color='red', linewidth=1, linestyle='--', alpha=0.5)
ax2.text(len(q2_3) - 1, 81, '80% threshold', color='red', fontsize=8, ha='right')
ax2.set_ylabel('Cumulative % of Revenue', color='red')
ax2.set_ylim(0, 110)

legend_handles = [mpatches.Patch(facecolor=v, label=k) for k, v in cat_colors.items()]
ax1.legend(handles=legend_handles, title='Category', loc='upper left')
plt.tight_layout()
plt.show()

top80 = q2_3[q2_3['cumulative_pct'] <= 80]
print(f'\n➡ {len(top80)} of {len(q2_3)} sub-categories account for 80% of revenue.')
print(f'  Key sub-categories: {", ".join(top80["sub_category"].tolist())}')

### Query 2.4 — Sub-Category Revenue Rank within Each Category
*Uses `DENSE_RANK() OVER (PARTITION BY ...)` to rank sub-categories within their parent category*

In [ ]:
q2_4 = run_query("""
    WITH sub_rev AS (
        SELECT
            p.category,
            p.sub_category,
            ROUND(SUM(o.sales), 2) AS total_revenue
        FROM orders o
        JOIN products p ON o.product_id = p.product_id
        GROUP BY p.category, p.sub_category
    )
    SELECT
        category,
        sub_category,
        total_revenue,
        DENSE_RANK() OVER (
            PARTITION BY category
            ORDER BY total_revenue DESC
        ) AS rank_in_category
    FROM sub_rev
    ORDER BY category, rank_in_category
""")

print('Sub-Category Revenue Ranking within Each Category:')
print(q2_4.to_string(index=False))

---
## 👥 Section 3 — Customer Analysis
> *Retention is more cost-effective than acquisition. This section identifies loyalty patterns and flags at-risk customers.*
---
### Query 3.1 — Top 10 Customers by Revenue

In [ ]:
q3_1 = run_query("""
    SELECT
        c.customer_name,
        c.segment,
        c.region,
        COUNT(DISTINCT o.order_id)  AS total_orders,
        ROUND(SUM(o.sales), 2)      AS total_revenue,
        MIN(o.order_date)           AS first_order,
        MAX(o.order_date)           AS last_order
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.customer_name, c.segment, c.region
    ORDER BY total_revenue DESC
    LIMIT 10
""")

print('Top 10 Customers by Revenue:')
print(q3_1.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(q3_1['customer_name'][::-1], q3_1['total_revenue'][::-1],
               color=sns.color_palette('Blues_d', len(q3_1)), edgecolor='white')
ax.set_title('Top 10 Customers by Total Revenue', fontsize=13, fontweight='bold')
ax.set_xlabel('Total Revenue ($)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for bar, val in zip(bars, q3_1['total_revenue'][::-1]):
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

### Query 3.2 — New vs. Returning Customers by Year

In [ ]:
q3_2 = run_query("""
    WITH first_purchase AS (
        SELECT customer_id,
               MIN(strftime('%Y', order_date)) AS first_year
        FROM orders
        GROUP BY customer_id
    ),
    yearly AS (
        SELECT o.customer_id,
               strftime('%Y', o.order_date) AS order_year,
               f.first_year
        FROM orders o
        JOIN first_purchase f ON o.customer_id = f.customer_id
    )
    SELECT
        order_year,
        COUNT(DISTINCT CASE WHEN order_year = first_year THEN customer_id END) AS new_customers,
        COUNT(DISTINCT CASE WHEN order_year > first_year  THEN customer_id END) AS returning_customers
    FROM yearly
    GROUP BY order_year
    ORDER BY order_year
""")

print('New vs. Returning Customers by Year:')
print(q3_2.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
x = range(len(q3_2))
width = 0.35
ax.bar([xi - width/2 for xi in x], q3_2['new_customers'],        width, label='New Customers',       color='#4C72B0', edgecolor='white')
ax.bar([xi + width/2 for xi in x], q3_2['returning_customers'],  width, label='Returning Customers', color='#55A868', edgecolor='white')
ax.set_title('New vs. Returning Customers by Year', fontsize=13, fontweight='bold')
ax.set_xticks(list(x))
ax.set_xticklabels(q3_2['order_year'])
ax.set_ylabel('Number of Customers')
ax.legend()
plt.tight_layout()
plt.show()

---
## ⭐ Section 4 — RFM Customer Segmentation *(Differentiator)*

> **RFM (Recency, Frequency, Monetary) is an industry-standard framework used by retail analytics teams to segment customers and prioritize marketing spend. This section builds a complete RFM scoring engine entirely in SQL using multi-step CTEs and the `NTILE()` window function.**

| Dimension | Definition | Why It Matters |
|---|---|---|
| **Recency (R)** | Days since the customer's last order | Recent buyers are more likely to buy again |
| **Frequency (F)** | Number of distinct orders placed | Frequent buyers are more loyal |
| **Monetary (M)** | Total revenue generated | High spenders are the most valuable customers |

Each customer receives a score of 1–4 on each dimension via `NTILE(4)`. The combined score determines their segment.

---
### Query 4.1 — Build the Full RFM Model in SQL

In [ ]:
rfm_query = """
    -- Step 1: Compute raw RFM metrics per customer
    WITH rfm_base AS (
        SELECT
            o.customer_id,
            c.customer_name,
            c.segment,
            c.region,
            CAST(
                julianday('2018-12-31') - julianday(MAX(o.order_date))
            AS INTEGER)                     AS recency_days,
            COUNT(DISTINCT o.order_id)      AS frequency,
            ROUND(SUM(o.sales), 2)          AS monetary
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        GROUP BY o.customer_id, c.customer_name, c.segment, c.region
    ),
    -- Step 2: Score each dimension using NTILE(4)
    -- Recency:  fewer days = better = higher score (inverted with 5-NTILE trick)
    -- Frequency & Monetary: higher values = higher score
    rfm_scored AS (
        SELECT *,
            5 - NTILE(4) OVER (ORDER BY recency_days DESC) AS r_score,
            NTILE(4)     OVER (ORDER BY frequency ASC)     AS f_score,
            NTILE(4)     OVER (ORDER BY monetary ASC)      AS m_score
        FROM rfm_base
    ),
    -- Step 3: Classify each customer into a named segment
    rfm_segmented AS (
        SELECT *,
            (r_score + f_score + m_score) AS rfm_total,
            CASE
                WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4  THEN 'Champions'
                WHEN r_score >= 3 AND f_score >= 3                   THEN 'Loyal Customers'
                WHEN r_score >= 4 AND f_score <= 2                   THEN 'New Customers'
                WHEN r_score <= 2 AND f_score >= 3 AND m_score >= 3  THEN 'At Risk'
                WHEN r_score <= 2 AND f_score <= 2                   THEN 'Lost'
                ELSE                                                      'Potential Loyalists'
            END AS rfm_segment
        FROM rfm_scored
    )
    SELECT * FROM rfm_segmented
    ORDER BY rfm_total DESC
"""

rfm_full = run_query(rfm_query)
print(f'RFM scores computed for {len(rfm_full):,} customers.')
print('\nTop 10 customers by RFM score:')
print(rfm_full.head(10)[['customer_name','segment','region',
                           'recency_days','frequency','monetary',
                           'r_score','f_score','m_score',
                           'rfm_total','rfm_segment']].to_string(index=False))

### Query 4.2 — RFM Segment Summary

In [ ]:
q4_2 = run_query("""
    WITH rfm_base AS (
        SELECT o.customer_id, c.customer_name, c.segment,
            CAST(julianday('2018-12-31') - julianday(MAX(o.order_date)) AS INTEGER) AS recency_days,
            COUNT(DISTINCT o.order_id)  AS frequency,
            ROUND(SUM(o.sales), 2)      AS monetary
        FROM orders o JOIN customers c ON o.customer_id = c.customer_id
        GROUP BY o.customer_id, c.customer_name, c.segment
    ),
    rfm_scored AS (
        SELECT *,
            5 - NTILE(4) OVER (ORDER BY recency_days DESC) AS r_score,
            NTILE(4)     OVER (ORDER BY frequency ASC)     AS f_score,
            NTILE(4)     OVER (ORDER BY monetary ASC)      AS m_score
        FROM rfm_base
    ),
    rfm_segmented AS (
        SELECT *,
            CASE
                WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN 'Champions'
                WHEN r_score >= 3 AND f_score >= 3                  THEN 'Loyal Customers'
                WHEN r_score >= 4 AND f_score <= 2                  THEN 'New Customers'
                WHEN r_score <= 2 AND f_score >= 3 AND m_score >= 3 THEN 'At Risk'
                WHEN r_score <= 2 AND f_score <= 2                  THEN 'Lost'
                ELSE                                                     'Potential Loyalists'
            END AS rfm_segment
        FROM rfm_scored
    )
    SELECT
        rfm_segment,
        COUNT(*)                        AS customer_count,
        ROUND(AVG(recency_days), 0)     AS avg_recency_days,
        ROUND(AVG(frequency), 1)        AS avg_orders,
        ROUND(AVG(monetary), 2)         AS avg_revenue,
        ROUND(SUM(monetary), 2)         AS total_revenue
    FROM rfm_segmented
    GROUP BY rfm_segment
    ORDER BY avg_revenue DESC
""")

print('RFM Segment Summary:')
print(q4_2.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('RFM Customer Segmentation Results', fontsize=14, fontweight='bold')
palette = sns.color_palette('Set2', len(q4_2))

axes[0].pie(q4_2['customer_count'], labels=q4_2['rfm_segment'],
            autopct='%1.0f%%', colors=palette, startangle=90)
axes[0].set_title('Customer Count by Segment')

bars = axes[1].bar(q4_2['rfm_segment'], q4_2['avg_revenue'], color=palette, edgecolor='white')
axes[1].set_title('Avg Revenue per Customer')
axes[1].set_ylabel('Avg Revenue ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].tick_params(axis='x', rotation=20)
for bar, val in zip(bars, q4_2['avg_revenue']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'${val:,.0f}', ha='center', fontsize=8)

bars2 = axes[2].bar(q4_2['rfm_segment'], q4_2['total_revenue'], color=palette, edgecolor='white')
axes[2].set_title('Total Revenue by Segment')
axes[2].set_ylabel('Total Revenue ($)')
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[2].tick_params(axis='x', rotation=20)
for bar, val in zip(bars2, q4_2['total_revenue']):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'${val:,.0f}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

### Query 4.3 — Targeted Marketing Action Plan per Segment

In [ ]:
actions = {
    'Champions':           'Reward them. Ask for reviews. Offer early access to new products.',
    'Loyal Customers':     'Upsell higher-value products. Offer loyalty rewards and volume discounts.',
    'Potential Loyalists': 'Offer membership programs. Send personalized product recommendations.',
    'New Customers':       'Send onboarding emails. Educate on product range and policies.',
    'At Risk':             'Send win-back campaigns. Offer discounts on previously purchased items.',
    'Lost':                'Re-engage with strong incentives or reallocate marketing spend elsewhere.'
}

print('RFM-Based Marketing Action Plan')
print('=' * 95)
for _, row in q4_2.iterrows():
    action = actions.get(row['rfm_segment'], 'Review individually.')
    print(f"\n[{row['rfm_segment']}]")
    print(f"  Customers: {row['customer_count']}  |  Avg Revenue: ${row['avg_revenue']:,.2f}  |  Avg Days Since Last Order: {row['avg_recency_days']:.0f}")
    print(f"  → {action}")

---
## 🚚 Section 5 — Shipping & Operations
> *Shipping speed directly impacts customer satisfaction. This section evaluates delivery performance by mode and region.*
---
### Query 5.1 — Shipping Mode Performance

In [ ]:
q5_1 = run_query("""
    SELECT
        ship_mode,
        COUNT(DISTINCT order_id)                                AS total_orders,
        ROUND(COUNT(DISTINCT order_id) * 100.0
              / SUM(COUNT(DISTINCT order_id)) OVER (), 1)       AS pct_of_orders,
        ROUND(AVG(days_to_ship), 1)                             AS avg_days_to_ship,
        MIN(days_to_ship)                                       AS min_days,
        MAX(days_to_ship)                                       AS max_days,
        ROUND(SUM(sales), 2)                                    AS total_revenue
    FROM orders
    GROUP BY ship_mode
    ORDER BY avg_days_to_ship
""")

print('Shipping Mode Performance:')
print(q5_1.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Shipping Mode Analysis', fontsize=14, fontweight='bold')
palette = sns.color_palette('Set2', 4)

axes[0].pie(q5_1['total_orders'], labels=q5_1['ship_mode'],
            autopct='%1.1f%%', colors=palette, startangle=90)
axes[0].set_title('Order Share by Shipping Mode')

bars = axes[1].bar(q5_1['ship_mode'], q5_1['avg_days_to_ship'], color=palette, edgecolor='white')
axes[1].set_title('Average Days to Ship by Mode')
axes[1].set_ylabel('Avg Days to Ship')
axes[1].tick_params(axis='x', rotation=15)
for bar, val in zip(bars, q5_1['avg_days_to_ship']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{val} days', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

### Query 5.2 — Shipping Speed Heatmap: Region × Ship Mode
*Are customers in some regions waiting longer than others for the same shipping mode?*

In [ ]:
q5_2 = run_query("""
    SELECT
        c.region,
        o.ship_mode,
        COUNT(DISTINCT o.order_id)    AS total_orders,
        ROUND(AVG(o.days_to_ship), 1) AS avg_days_to_ship
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.region, o.ship_mode
    ORDER BY c.region, avg_days_to_ship
""")

print('Avg Days to Ship by Region and Ship Mode:')
print(q5_2.to_string(index=False))

pivot = q5_2.pivot_table(index='region', columns='ship_mode', values='avg_days_to_ship')
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Avg Days to Ship'})
ax.set_title('Avg Days to Ship — Region × Shipping Mode', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📋 Section 6 — Key Findings & Business Recommendations

### 📈 Key Findings

| # | Theme | Finding |
|---|-------|---------|
| 1 | **Sales** | West region generates the highest revenue; all regions show consistent YoY growth 2015–2018 |
| 2 | **Sales** | Q4 is consistently the strongest quarter — driven by holiday purchasing patterns |
| 3 | **Sales** | Corporate segment generates the highest revenue per customer |
| 4 | **Products** | Technology dominates revenue; Phones and Chairs are the top sub-categories |
| 5 | **Products** | A small number of sub-categories account for ~80% of total revenue (Pareto principle) |
| 6 | **RFM** | Champions and Loyal Customers generate disproportionately high average revenue |
| 7 | **RFM** | A significant share of customers are Lost or At Risk — a critical retention gap |
| 8 | **Shipping** | Standard Class is used for the majority of orders; shipping times are consistent across regions |

---

### 💡 Business Recommendations

**1. Protect and reward Champions**
> Champions generate the highest average revenue. Offer exclusive early access, loyalty rewards, and personalized outreach to keep them engaged.

**2. Launch win-back campaigns for At Risk and Lost customers**
> These segments have high historical spend but haven't purchased recently. Targeted discounts or personalized re-engagement emails could recover significant revenue.

**3. Double down on Technology**
> Technology drives the highest revenue per order. Expanding accessories and phones range would amplify this further across all regions.

**4. Focus inventory investment on the top 80% sub-categories**
> The Pareto analysis shows a small number of sub-categories drive the vast majority of sales. Stock depth and marketing spend should reflect this concentration.

**5. Promote Standard Class upgrade paths**
> Most customers default to Standard Class. A well-placed upsell prompt at checkout for First Class could improve satisfaction and generate premium shipping revenue.

---

### 🔮 Future Work
- **Cohort analysis** — track revenue retention by customer acquisition year
- **Market basket analysis** — identify products frequently bought together for bundling
- **Predictive CLV model** — use RFM scores as features in a regression to forecast future customer value
- **Power BI dashboard** — bring these SQL findings to life in an interactive executive dashboard

In [ ]:
conn.close()
print('Analysis complete. Database connection closed.')